# 演習③ RAG パイプラインを組み立てる（穴埋め形式）

**対応ノートブック**: `notebooks/ch2_03_rag.ipynb`
**推奨所要時間**: 約 30 分

`# TODO:` と書かれた箇所を自分で埋めてください。
各課題の最後には **自己チェック**（`assert`）が入っています。`✅ OK` が出れば正解です。

わからない場合は、講師に声をかけてください。

---

## セットアップ（完成済み）

In [ ]:
# 実行時間: 数秒
import os
os.environ.setdefault('ANONYMIZED_TELEMETRY', 'False')   # ChromaDB の利用統計の送信を止める（当日はオフライン）

import unicodedata
from pathlib import Path

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

print('キャッシュ先:', os.environ.get('HF_HOME', '~/.cache/huggingface'))

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'使用デバイス: {device}')

# 検索対象の文書（chapter2/data/rag_docs）。ノートブックの場所から相対的に探す
DOCS_DIR = Path('../data/rag_docs')
if not DOCS_DIR.exists():
    DOCS_DIR = Path('chapter2/data/rag_docs')
print(f'文書の場所: {DOCS_DIR.resolve()}')


def pad(s, width: int) -> str:
    """全角文字を2文字ぶんとして数え、表示幅をそろえる（表の桁ずれ防止）"""
    s = str(s)
    w = sum(2 if unicodedata.east_asian_width(c) in 'WF' else 1 for c in s)
    return s + ' ' * max(width - w, 0)

In [ ]:
# 実行時間: 約30秒
LLM_NAME = 'meta-llama/Llama-3.2-1B-Instruct'

tokenizer = AutoTokenizer.from_pretrained(LLM_NAME, clean_up_tokenization_spaces=False)
llm = AutoModelForCausalLM.from_pretrained(
    LLM_NAME,
    dtype=torch.float16 if device == 'cuda' else torch.float32,
    device_map='auto',
)
llm.eval()
print(f'{LLM_NAME} を読み込みました')


def ask_llm(messages: list[dict], max_new_tokens: int = 200) -> str:
    """チャット形式のメッセージを渡し、LLM の回答だけを返す（貪欲法なので毎回同じ答えになる）"""
    inputs = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, return_tensors='pt', return_dict=True,
    ).to(llm.device)
    with torch.no_grad():
        out = llm.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(out[0, inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip()

In [ ]:
# 実行時間: 約10秒
EMB_NAME = 'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2'

embeddings = HuggingFaceEmbeddings(
    model_name=EMB_NAME,
    model_kwargs={'device': device},
    encode_kwargs={'normalize_embeddings': True},   # 長さ 1 にそろえる（内積 = コサイン類似度になる）
)
print(f'{EMB_NAME} を読み込みました')

In [ ]:
# 実行時間: 数秒
documents = [
    Document(page_content=p.read_text(encoding='utf-8'), metadata={'source': p.name})
    for p in sorted(DOCS_DIR.glob('*.md'))
    if p.name != 'README.md'
]

for d in documents:
    n_tokens = len(tokenizer.encode(d.page_content, add_special_tokens=False))
    print(f'  {pad(d.metadata["source"], 34)} {len(d.page_content):>6,} 文字  {n_tokens:>6,} トークン')

---

## 課題 1: 文書をチャンクに分割する

`RecursiveCharacterTextSplitter.from_huggingface_tokenizer()` で、次の条件の splitter を作ってください。

- トークン数は `tokenizer`（Llama のトークナイザ）で数える
- `chunk_size=256`、`chunk_overlap=32`
- `separators=SEPARATORS`

In [ ]:
SEPARATORS = ['\n\n', '\n', '。', '、', ' ', '']

# TODO: splitter を作ってください
splitter = None

# TODO: splitter で documents を分割してください（split_documents を使う）
chunks = None

print(f'チャンク数: {len(chunks)}')

# ── 自己チェック ──
max_tokens = max(len(tokenizer.encode(c.page_content, add_special_tokens=False)) for c in chunks)
assert len(chunks) > len(documents), '文書より多くのチャンクに分割されるはずです'
assert max_tokens <= 256, f'256 トークンを超えるチャンクがあります（最大 {max_tokens}）。トークナイザで数えていますか？'
assert all('source' in c.metadata for c in chunks), 'チャンクに出典（metadata の source）が残っていません'
print('✅ OK')

---

## 課題 2: ベクトル DB に入れて検索する

1. `Chroma` を作り、`chunks` を入れてください
   - `collection_name='ex-rag'`、`embedding_function=embeddings`
   - `collection_metadata={'hnsw:space': 'cosine'}`（距離 = 1 − コサイン類似度）
   - 作ったら `reset_collection()` で空にしてから `add_documents()` で入れる（セルを再実行しても二重に入らないようにするため）
2. `similarity_search_with_score()` で、質問に近い上位 3 件を取ってください

In [ ]:
from langchain_chroma import Chroma

# TODO: Chroma を作り、空にしてから chunks を入れてください
vectorstore = None

query = 'OutOfMemoryError が出たらどうすればよいですか？'

# TODO: 上位 3 件を (Document, 距離) のリストで取ってください
results = None

for i, (doc, dist) in enumerate(results):
    print(f'[{i + 1}] 距離={dist:.3f}  {doc.metadata["source"]}  {doc.page_content[:50]!r}')
print('\n→ 「batch size などを小さくして」と書かれたチャンクが入っているか、目で確かめましょう')

# ── 自己チェック ──
assert vectorstore._collection.count() == len(chunks), 'ベクトル DB の件数がチャンク数と一致しません（二重に入っていませんか）'
assert len(results) == 3, '3 件取ってください'
dists = [d for _, d in results]
assert dists == sorted(dists), '距離の小さい順（近い順）に並んでいるはずです'
print('✅ OK')

---

## 課題 3: RAG のプロンプトを組み立てる

検索したチャンクを【参考文書】としてプロンプトに貼り付ける関数を完成させてください。

- `system` メッセージに `SYSTEM_PROMPT` を入れる
- `user` メッセージは `【参考文書】\n{チャンクを区切り線でつないだもの}\n\n【質問】\n{質問}` の形にする
- チャンクは `'\n\n---\n\n'` でつなぐ

In [ ]:
SYSTEM_PROMPT = (
    'あなたはハンズオンの質問に答えるアシスタントです。'
    '【参考文書】に書かれている内容だけを使って、日本語で簡潔に答えてください。'
    '参考文書に答えがない場合は「参考文書には書かれていません」と答えてください。'
)


def build_rag_messages(query: str, docs: list) -> list[dict]:
    # TODO: チャンクの本文（page_content）を区切り線でつないでください
    context = None

    # TODO: system と user の 2 つのメッセージを返してください
    return None


def rag_answer(query: str, vectorstore, top_k: int = 3):
    docs = vectorstore.similarity_search(query, k=top_k)
    return ask_llm(build_rag_messages(query, docs)), docs


answer, docs = rag_answer(query, vectorstore)
print(f'【質問】{query}\n【回答】{answer}')

# ── 自己チェック ──
msgs = build_rag_messages('質問A', [Document(page_content='文書X'), Document(page_content='文書Y')])
assert [m['role'] for m in msgs] == ['system', 'user'], 'system → user の順の 2 メッセージにしてください'
assert msgs[0]['content'] == SYSTEM_PROMPT
assert msgs[1]['content'] == '【参考文書】\n文書X\n\n---\n\n文書Y\n\n【質問】\n質問A', '貼り付けの形式が違います'
print('✅ OK')

---

## 課題 4: 当たり率を計算する

上位 k 件のチャンクのどれかに正解の文字列が入っていた質問の割合（当たり率）を返す関数を完成させてください。

In [ ]:
# 検索の正解判定に使う質問と、答えが書かれている箇所に含まれる文字列
RETRIEVAL_TESTS = [
    ('何も実行せずに放置すると、カーネルはどうなりますか？',        '30分何も実行しないと'),
    ('SFT の学習を始めたのに待たされるのはなぜですか？',            '同時に走らせられる人数は限られています'),
    ('VS Code の Remote-SSH で DGX に接続できますか？',             'VS Code の Remote-SSH では DGX に接続できません'),
    ('OutOfMemoryError が出たらどうすればよいですか？',             'batch size などを小さくして'),
    ('教材ページを JupyterLab の中で開いてはいけないのはなぜですか？', '数式やリンクが正しく表示されない'),
    ('過学習を防ぐ方法を教えてください。',                          'ドロップアウト'),
    ('LoRA では何を学習しますか？',                                 '低ランク行列'),
]


def hit_rate(vs, k: int) -> float:
    # TODO: RETRIEVAL_TESTS の各質問で上位 k 件を検索し、正解の文字列を含むチャンクがあれば当たりとする
    # TODO: 当たりの数 / 質問数 を返す
    return None


for k in [1, 3, 5]:
    print(f'k={k}: 当たり率 {hit_rate(vectorstore, k):.0%}')

# ── 自己チェック ──
r1, r3, r5 = (hit_rate(vectorstore, k) for k in [1, 3, 5])
assert 0 <= r1 <= r3 <= r5 <= 1, 'k を増やしても当たり率は下がらないはずです'
assert r5 > 0, 'k=5 でも 1 問も当たっていません。判定の仕方を確認してください'
print('✅ OK')

---

## チャレンジ（任意）

1. `chunk_size` を 128 / 256 / 512 / 1024 に変え、`hit_rate(k=3)` の表を作ってください。当たり率が最も高い大きさはどれでしたか？ それはなぜだと考えますか？
2. `RETRIEVAL_TESTS` に、参加者手順書から自分で質問と正解の文字列を 3 組追加し、当たり率を測り直してください
3. 参加者手順書に書かれていないこと（例: 「昼休みは何時からですか？」）を `rag_answer()` で聞いてください。「参考文書には書かれていません」と答えましたか？